# Instrument Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Render a generic instrument valuation result with pricing assumptions, PV, metrics, and cash-flow context.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`.

**What you'll learn:**

- Prepare valuation inputs for one instrument.
- Render `reporting.instrument_tearsheet`.
- Keep pricing logic in the valuation API and presentation in reporting.

An **instrument tear sheet** is a single-page HTML report for a priced instrument:
headline KPIs (NPV, yield, duration, DV01), a definition block, per-bucket key-rate
sensitivities, a cashflow ladder, and a scrollable cashflow schedule.

`reporting.instrument_tearsheet(result, ...)` returns a `TearSheet` whose `_repr_html_`
method lets Jupyter display it inline — just make it the last expression in a cell.
Use `reporting.recommended_metrics(instrument_type)` to get the right metric IDs to pass
to `price_instrument_with_metrics`.


In [ ]:
import datetime as dt
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.valuations import ValuationResult, instrument_cashflows
from finstack_quant.valuations.instruments import price_instrument_with_metrics
from finstack_quant import reporting

In [ ]:
# Build a USD fixed-rate bond: ACME 4.25% maturing 2034-03-15.
bond = json.dumps({"type": "bond", "spec": {
    "id": "ACME 4.25% 2034",
    "notional": {"amount": "10000000", "currency": "USD"},
    "issue_date": "2024-03-15",
    "maturity": "2034-03-15",
    "cashflow_spec": {"Fixed": {
        "coupon_type": "Cash", "rate": 0.0425,
        "freq": {"count": 6, "unit": "months"},
        "dc": "Thirty360", "bdc": "following",
        "calendar_id": "weekends_only", "stub": "None",
        "end_of_month": False, "payment_lag_days": 0}},
    "discount_curve_id": "USD-OIS",
    "call_put": None,
    "attributes": {"tags": [], "meta": {}},
    "pricing_overrides": {}}})

# USD-OIS discount curve anchored to 2026-06-19.
mc = MarketContext()
mc.insert(DiscountCurve(
    "USD-OIS", date(2026, 6, 19),
    [(0.0, 1.0), (0.5, 0.98), (1.0, 0.96), (2.0, 0.92),
     (3.0, 0.88), (5.0, 0.80), (10.0, 0.65)],
    day_count="act_365f"))

as_of = "2026-06-19"
print(f"Bond: {json.loads(bond)['spec']['id']}")
print(f"As-of: {as_of}")

In [ ]:
# Price with the recommended metric set for a full bond sheet.
# ytw, oas, asw_par require a market-quoted clean price in pricing_overrides;
# drop those so this notebook runs without extra market data.
_NEEDS_QUOTE = {"ytw", "oas", "asw_par"}
metrics = [m for m in reporting.recommended_metrics("bond") if m not in _NEEDS_QUOTE]

result = ValuationResult.from_json(
    price_instrument_with_metrics(bond, mc.to_json(), as_of, model="discounting", metrics=metrics))
_, cashflows = instrument_cashflows(bond, mc.to_json(), as_of, model="discounting")

print("Price (NPV):", result.price, result.currency)
print("YTM:       ", result.get_metric("ytm"))
print("DV01:      ", result.get_metric("dv01"))

In [ ]:
# Full tear sheet — rendered inline via _repr_html_.
# Native <title> tooltips work here; open the saved .html in a browser for the
# richer crosshair + cursor-following tooltip.
reporting.instrument_tearsheet(
    result,
    definition=json.loads(bond),
    cashflows=cashflows,
    generated=dt.date(2026, 6, 19),
)

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
